# 02 - Classical Baselines & Quantum Models (Tasks 2-4)

Thin interactive wrapper around `src.classical_models`, `src.quantum_models`, and `src.evaluation`.

Every quantum computation below runs as a real `SamplerV2` job (see `src.quantum_backend`). By default it targets the local `AerSimulator`; set `EXECUTION_CONFIG = ExecutionConfig(mode="ibm_runtime")` (after running `python scripts/setup_ibm_account.py`) to route the same code at real IBM Quantum hardware.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import CLASSIFICATION_TARGET, MODELING_TABLE_CSV
from src.features import get_candidate_matrix, select_k_best_features
from src.classical_models import run_classical_classification_suite
from src.quantum_backend import ExecutionConfig, QuantumExecutor
from src.quantum_models import run_quantum_classification_suite, compute_quantum_kernel_matrix
from src.evaluation import summarize_results, kernel_target_alignment

EXECUTION_CONFIG = ExecutionConfig(mode="aer_simulator", shots=4096)

data = pd.read_csv(MODELING_TABLE_CSV)
X_full = get_candidate_matrix(data)
y = data[CLASSIFICATION_TARGET]
selection = select_k_best_features(X_full, y)
X = X_full[selection["selected_features"]]
selection["selected_features"]

In [ ]:
clf_results = run_classical_classification_suite(X, y)
summarize_results(clf_results)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
X_scaled = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X.values)
kernel_executor = QuantumExecutor(EXECUTION_CONFIG)
K = compute_quantum_kernel_matrix(X_scaled, kernel_executor, feature_map_name="angle")
print("KTA (angle feature map):", kernel_target_alignment(K, y.values))

In [ ]:
# Full 29-fold LOOCV over every registered quantum model. On ibm_runtime this
# submits one Sampler job per fold per model -- consider slicing X.values[:8],
# y.values[:8] first for a quick smoke test against real hardware.
quantum_results = run_quantum_classification_suite(X.values, y.values, execution_config=EXECUTION_CONFIG)
summarize_results(quantum_results)